# Haptic Ground Truth — Colab Pipeline

Convert **3–5 minute** video audio into **four candidate haptic tracks** (Algorithms A–D) for human-in-the-loop evaluation.

**Runtime:** CPU only — no GPU required.

| Algorithm | Method |
|-----------|--------|
| **A** | Perception mapping (loudness + roughness → dual sine) |
| **B** | Frequency shifting (-12/-24 semitones + 10–250 Hz bandpass) |
| **C** | Bark-band pitch tracking → variable sine |
| **D** | HapticGen (200 Hz ± 50 Hz NCO from RMS energy) |

In [ ]:
# Install system + Python dependencies (CPU runtime is fine)
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q numpy scipy librosa soundfile audioread

In [ ]:
import sys
from pathlib import Path

# Option 1: clone from GitHub (after you push this repo)
# !git clone https://github.com/YOUR_USER/haptic-groundtruth.git
# PROJECT_ROOT = Path('/content/haptic-groundtruth')

# Option 2: upload project zip to /content and unzip
# from google.colab import files
# uploaded = files.upload()  # upload haptic-groundtruth.zip
# !unzip -q haptic-groundtruth.zip -d /content
# PROJECT_ROOT = Path('/content/haptic-groundtruth')

# Option 3: mount Drive if project lives there
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = Path('/content/drive/MyDrive/haptic-groundtruth')

# Default: assume repo uploaded/unzipped to /content/haptic-groundtruth
PROJECT_ROOT = Path('/content/haptic-groundtruth')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        'Project not found. Upload/unzip haptic-groundtruth to /content, '
        'or edit PROJECT_ROOT above.'
    )

sys.path.insert(0, str(PROJECT_ROOT))
print('Using project root:', PROJECT_ROOT)

In [ ]:
from google.colab import files
from IPython.display import Audio, display
from haptic_gt.pipeline import generate_candidate_tracks

OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Upload a video file (mp4, mov, mkv, ...). Max ~5 min recommended.')
uploaded = files.upload()
video_name = next(iter(uploaded))
video_path = Path('/content') / video_name
print('Uploaded:', video_path)

In [ ]:
%%time
tracks = generate_candidate_tracks(
    video_path,
    OUTPUT_DIR,
    from_video=True,
    target_rms=0.1,
)
saved = tracks.save_all()

duration_sec = len(tracks.source_audio) / tracks.sample_rate
print(f'Duration: {duration_sec:.1f}s @ {tracks.sample_rate} Hz')
print('Saved files:')
for name, path in saved.items():
    print(f'  {name}: {path}')

In [ ]:
labels = {
    'source_audio': 'Source audio',
    'algorithm_a_perception_mapping': 'A — Perception mapping',
    'algorithm_b_frequency_shifting': 'B — Frequency shifting',
    'algorithm_c_pitch_matching': 'C — Pitch matching',
    'algorithm_d_haptic_gen': 'D — HapticGen',
}

for key, label in labels.items():
    print(label)
    display(Audio(str(saved[key])))

In [ ]:
import shutil

zip_path = shutil.make_archive('/content/haptic_candidates', 'zip', OUTPUT_DIR)
files.download(zip_path)
print('Downloaded:', zip_path)

## Human-in-the-loop (next step)

1. Play **source audio** while feeling each candidate on haptic hardware.
2. Rate **realism** and **similarity** (e.g. 1–7 Likert) per algorithm.
3. If evaluator agreement < threshold, tune parameters in `haptic_gt/algorithm_*.py` and re-run.
4. Approved tracks become your **ground truth dataset**.